# Experiment 008 — KL-Selective JRR

Проверяем, достаточно ли удалять только KL-increasing component nonlinear JRR remainder. Старый JRR held-out не переиспользуется как confirmatory data; новый split был заморожен заранее.


In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/content/steering-manifold-repair')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/Nek1tt/steering-manifold-repair.git',str(repo)], check=True)
else:
    subprocess.run(['git','-C',str(repo),'pull','--ff-only'], check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'], check=True)
print('cwd:', os.getcwd())


## 0. Восстановить frozen sentiment direction при необходимости

In [ ]:
from pathlib import Path
direction = Path('results/sentiment_direction.pt')
if not direction.exists():
    subprocess.run([sys.executable,'scripts/validate_sentiment_baseline.py','--config','configs/baseline_sentiment_gpt2.yaml'], check=True)
else:
    print('Using frozen direction:', direction)


## 1. Unit tests

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_selective_jrr.py','tests/test_jrr.py'], check=True)

## 2. Real-model gradient preflight
Проверяем KL gradient независимой finite-difference derivative и убеждаемся, что выбранная correction ортогональна transported `Jv`.


In [ ]:
subprocess.run([sys.executable,'scripts/preflight_selective_jrr.py','--config','configs/selective_jrr_gpt2.yaml'], check=True)

## 3. Calibration
Сравниваем additive, full JRR и KL-JRR только на calibration prompts. Layer и `beta` не sweep-ятся.


In [ ]:
subprocess.run([sys.executable,'scripts/run_selective_jrr.py','--config','configs/selective_jrr_gpt2.yaml','--phase','calibration'], check=True)

In [ ]:
import json, pandas as pd
from IPython.display import display, Image
cal = json.loads(Path('results/selective_jrr/calibration_summary.json').read_text())
print(json.dumps(cal, indent=2))
display(pd.read_csv('results/selective_jrr/calibration_same_alpha.csv'))
display(pd.read_csv('results/selective_jrr/calibration_aggregate.csv'))
display(Image(filename='results/selective_jrr/calibration_pareto.png'))

## 4. Новый frozen held-out
Используются 12 новых prompts и seeds 101/211. Evaluation выполняется только если заранее зафиксированный calibration gate пройден. В финальном run gate не прошёл, поэтому held-out должен остаться нетронутым.


In [ ]:
if cal['go_to_new_heldout']:
    subprocess.run([sys.executable,'scripts/run_selective_jrr.py','--config','configs/selective_jrr_gpt2.yaml','--phase','evaluation'], check=True)
    print('Fresh held-out complete.')
else:
    print('STOP: calibration gate failed. Do not use --force for the reported result.')

In [ ]:
if cal['go_to_new_heldout']:
    ev = json.loads(Path('results/selective_jrr/evaluation_summary.json').read_text())
    print(json.dumps(ev, indent=2))
    display(pd.read_csv('results/selective_jrr/evaluation_same_alpha.csv'))
    display(pd.read_csv('results/selective_jrr/evaluation_aggregate.csv'))
    display(Image(filename='results/selective_jrr/evaluation_pareto.png'))

## 5. Упаковать runtime results
ZIP полезен для независимого анализа даже при failed calibration gate.


In [ ]:
import shutil
archive = shutil.make_archive('/content/selective_jrr_results','zip','results/selective_jrr')
print('Created:', archive)